# SQL basics (SQLite)

> **Note:** the notebooks in `examples/` are reset every time your server starts, so they are always up to date. To keep your changes, copy this notebook into `work/` (right-click it in the file browser → *Copy*, then paste it in `work/`).

This notebook uses the **xeus-sqlite** kernel: each cell is an **SQL** statement run on an SQLite database.
We'll build a small school database and ask it questions. Run a cell with **Shift + Enter**.

Kernel commands start with `%`: `%CREATE file.db` creates (or opens) a database, `%LOAD file.db rw` opens an existing one.

## 1. Create a database

We keep the database in `/tmp` so it doesn't clutter your files. Run every cell from the top so the database is built in order.

In [ ]:
%CREATE /tmp/school.db

In [ ]:
DROP TABLE IF EXISTS enrollments;

In [ ]:
DROP TABLE IF EXISTS students;

In [ ]:
DROP TABLE IF EXISTS courses;

## 2. Tables: `CREATE TABLE`

A **table** has columns, each with a type. A **primary key** identifies each row; a **foreign key** points to a row in another table.

In [ ]:
CREATE TABLE students (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL,
    age     INTEGER,
    city    TEXT
);

In [ ]:
CREATE TABLE courses (
    id      INTEGER PRIMARY KEY,
    title   TEXT NOT NULL,
    credits INTEGER
);

In [ ]:
CREATE TABLE enrollments (
    student_id INTEGER REFERENCES students(id),
    course_id  INTEGER REFERENCES courses(id),
    grade      REAL
);

## 3. Adding rows: `INSERT`

In [ ]:
INSERT INTO students (id, name, age, city) VALUES
    (1, 'Ada',   20, 'London'),
    (2, 'Alan',  22, 'Manchester'),
    (3, 'Grace', 21, 'New York'),
    (4, 'Linus', 19, 'Helsinki'),
    (5, 'Hedy',  23, 'Vienna');

In [ ]:
INSERT INTO courses (id, title, credits) VALUES
    (1, 'Programming', 6),
    (2, 'Databases',   6),
    (3, 'Networks',    4);

In [ ]:
INSERT INTO enrollments (student_id, course_id, grade) VALUES
    (1, 1, 9.5), (1, 2, 8.0),
    (2, 1, 7.0), (2, 3, 6.5),
    (3, 2, 9.0), (3, 3, 8.5),
    (4, 1, 4.5),
    (5, 2, 7.5);

## 4. Asking questions: `SELECT`

`SELECT columns FROM table` reads data. `*` means all columns.

In [ ]:
SELECT * FROM students;

### Filtering with `WHERE` and sorting with `ORDER BY`

Combine conditions with `AND` / `OR`. `LIKE 'A%'` matches text starting with A. `ORDER BY … DESC` sorts from high to low.

In [ ]:
SELECT name, age FROM students WHERE age >= 21 ORDER BY age DESC;

In [ ]:
SELECT name, city FROM students WHERE name LIKE 'A%' OR city = 'Vienna';

## 5. Aggregates and `GROUP BY`

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` summarise many rows into one. `GROUP BY` makes one summary per group;
`HAVING` filters the groups.

In [ ]:
SELECT COUNT(*) AS students, AVG(age) AS average_age, MIN(age) AS youngest FROM students;

In [ ]:
SELECT course_id, COUNT(*) AS enrolled, ROUND(AVG(grade), 2) AS average_grade
FROM enrollments
GROUP BY course_id
HAVING COUNT(*) >= 2;

## 6. Combining tables: `JOIN`

A `JOIN` matches rows from two tables using a condition, usually a foreign key = primary key.

In [ ]:
SELECT s.name, c.title, e.grade
FROM enrollments AS e
JOIN students AS s ON s.id = e.student_id
JOIN courses  AS c ON c.id = e.course_id
ORDER BY s.name, c.title;

In [ ]:
SELECT c.title, ROUND(AVG(e.grade), 2) AS average_grade, COUNT(*) AS students
FROM courses AS c
JOIN enrollments AS e ON e.course_id = c.id
GROUP BY c.title
ORDER BY average_grade DESC;

A `LEFT JOIN` keeps every row from the left table, even if there's no match (the missing values are `NULL`).

In [ ]:
SELECT s.name, COUNT(e.course_id) AS courses
FROM students AS s
LEFT JOIN enrollments AS e ON e.student_id = s.id
GROUP BY s.name;

## 7. Changing data: `UPDATE` and `DELETE`

⚠️ Always use a `WHERE` clause, or the change applies to **every** row.

In [ ]:
UPDATE enrollments SET grade = 5.0 WHERE student_id = 4 AND course_id = 1;

In [ ]:
DELETE FROM enrollments WHERE grade < 7;

In [ ]:
SELECT * FROM enrollments;

## Try it yourself

1. List the students who live in a city that starts with `N` or `M`.
2. Show each student's name with their best grade.
3. Add a new course and enrol two students in it, then check the result with a `JOIN`.

In [ ]:
SELECT name FROM students;